In [3]:
# 1. Install & Import Library


!pip install tensorflow scikit-learn -q


import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import Callback, EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score


In [4]:
# 2. Upload & Load Dataset


from google.colab import files
import io

uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(df.shape)
df.head()


Saving Personal_Finance_Dataset_11500.csv to Personal_Finance_Dataset_11500.csv
(11500, 5)


,Date,Transaction Description,Category,Amount,Type
0,2020-01-02,Score each.,Food & Drink,1485.69,Expense
1,2020-01-02,Quality throughout.,Utilities,1475.58,Expense
2,2020-01-04,Instead ahead despite measure ago.,Rent,1185.08,Expense
3,2020-01-05,Information last everything thank serve.,Investment,2291.00,Income
4,2020-01-13,Future choice whatever from.,Food & Drink,1126.88,Expense


In [5]:
# 3. Feature Engineering


# Copy dataset
DF = df.copy()

# Convert tanggal
DF['Date'] = pd.to_datetime(DF['Date'])

# Feature waktu
DF['year']       = DF['Date'].dt.year
DF['month']      = DF['Date'].dt.month
DF['day']        = DF['Date'].dt.day
DF['dayofweek']  = DF['Date'].dt.dayofweek
DF['quarter']    = DF['Date'].dt.quarter
DF['is_weekend'] = DF['dayofweek'].isin([5,6]).astype(int)

# Encode type
DF['type_encoded'] = (DF['Type'] == 'Income').astype(int)

# Log transform amount
DF['log_amount'] = np.log1p(DF['Amount'])

# Label Encoder target
le = LabelEncoder()
DF['label'] = le.fit_transform(DF['Category'])

# Feature list
FEATURES = [
    'Amount',
    'log_amount',
    'type_encoded',
    'year',
    'month',
    'day',
    'dayofweek',
    'quarter',
    'is_weekend'
]

X = DF[FEATURES].values
y = DF['label'].values

print(X.shape)
print(y.shape)


(11500, 9)
(11500,)


In [6]:
# 4. Split Data & Scaling


X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print('Train :', X_train.shape)
print('Val   :', X_val.shape)
print('Test  :', X_test.shape)



Train : (8050, 9)
Val   : (1725, 9)
Test  : (1725, 9)


In [7]:
# 5. Custom Callback (Checklist Wajib)


class CustomTrainingCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        acc = logs.get('accuracy')
        val_acc = logs.get('val_accuracy')

        print(f"Epoch {epoch+1} selesai")
        print(f"Training Accuracy : {acc:.4f}")
        print(f"Validation Accuracy : {val_acc:.4f}")
        print('-' * 40)


In [8]:
# 6. Build Deep Learning Model (Functional API)


input_layer = Input(shape=(X_train.shape[1],))

x = Dense(128, activation='relu')(input_layer)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Dense(64, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Dense(32, activation='relu')(x)

output_layer = Dense(len(le.classes_), activation='softmax')(x)

model = Model(inputs=input_layer, outputs=output_layer)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 9)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         1,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 12)             │           396 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,780 (49.92 KB)

 Trainable params: 12,396 (48.42 KB)

 Non-trainable params: 384 (1.50 KB)

In [9]:
# 7. Compile Model

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [10]:
# 8. Training Model


early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[
        CustomTrainingCallback(),
        early_stop
    ]
)


Epoch 1/30
251/252 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1927 - loss: 2.2881Epoch 1 selesai
Training Accuracy : 0.2157
Validation Accuracy : 0.2301
----------------------------------------
252/252 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.2157 - loss: 2.0371 - val_accuracy: 0.2301 - val_loss: 1.7049
Epoch 2/30
251/252 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2389 - loss: 1.7403Epoch 2 selesai
Training Accuracy : 0.2369
Validation Accuracy : 0.2359
----------------------------------------
252/252 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.2369 - loss: 1.7308 - val_accuracy: 0.2359 - val_loss: 1.6185
Epoch 3/30
250/252 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2411 - loss: 1.6972Epoch 3 selesai
Training Accuracy : 0.2431
Validation Accuracy : 0.2528
----------------------------------------
252/252 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.2431 - loss: 1.6766 - val_accuracy: 0.2528 - val_loss: 1.6023
Epoch 4/30
248/252 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/st

In [11]:
# 9. Evaluasi Model


loss, acc = model.evaluate(X_test, y_test)

print(f"Test Accuracy : {acc:.4f}")



# Prediksi
pred_probs = model.predict(X_test)
pred = np.argmax(pred_probs, axis=1)

print(classification_report(y_test, pred, target_names=le.classes_))



54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2603 - loss: 1.5640
Test Accuracy : 0.2603
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
                  precision    recall  f1-score   support

   Entertainment       0.21      0.05      0.09       296
    Food & Drink       0.24      0.12      0.16       297
Health & Fitness       0.20      0.09      0.12        23
          Income       0.91      0.88      0.90        78
      Investment       0.00      0.00      0.00        21
           Other       0.48      0.80      0.60        20
            Rent       0.13      0.12      0.12        25
          Salary       0.70      0.72      0.71        64
        Shopping       0.21      0.14      0.17       297
       Transport       0.19      0.23      0.21       270
          Travel       0.14      0.38      0.20        24
       Utilities       0.21      0.47      0.29       310

        accuracy                           0.26      1725
       macro avg       0.30      0.33      0.30    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
# 10. Simpan Model (.keras dan SavedModel)


# Format Keras
model.save('personal_finance_model.keras')

# Format SavedModel
model.export('saved_model_personal_finance')

print('Model berhasil disimpan')


Saved artifact at 'saved_model_personal_finance'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 9), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 12), dtype=tf.float32, name=None)
Captures:
  138536142853456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138536142852112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138536142855568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138536142852880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138536142852688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138536142856528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138536142856720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138536142856912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138536142857680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138536142857872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13853614285

In [13]:
# 11. Kode Inference Sederhana


# Contoh data baru
sample_data = pd.DataFrame({
    'Amount': [250000],
    'log_amount': [np.log1p(250000)],
    'type_encoded': [0],
    'year': [2024],
    'month': [7],
    'day': [15],
    'dayofweek': [1],
    'quarter': [3],
    'is_weekend': [0]
})

# Scaling
sample_scaled = scaler.transform(sample_data)

# Prediksi
prediction = model.predict(sample_scaled)
pred_class = np.argmax(prediction, axis=1)[0]

# Decode hasil
hasil = le.inverse_transform([pred_class])[0]

print('Kategori Prediksi :', hasil)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Kategori Prediksi : Salary


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [14]:
# 12. Load Model Kembali


loaded_model = tf.keras.models.load_model('personal_finance_model.keras')

print('Model berhasil di-load kembali')


Model berhasil di-load kembali
